In [ ]:
# ✅ Kaggle Notebookで実行可能な、前処理強化＋3エポック学習の完全版（修正版: train_test_splitの型エラー対応）

# 1. ライブラリのインポート
import pandas as pd
import numpy as np
import tensorflow as tf
import re
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, TFDistilBertForSequenceClassification

# 2. データ読み込み
train_df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")

# 3. 前処理関数（強化版）
# 再調整例（有用な情報は残す）
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\\S+", "http", text)  # URLは「http」に置換（痕跡を残す）
    text = re.sub(r"@[\\w]+", "@user", text)  # メンションは統一
    text = re.sub(r"[^a-zA-Z0-9@#:\\s]", "", text)  # 記号は一部残す
    return text.strip()


train_df["text"] = train_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

# 4. トークナイザーとエンコード
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
train_encodings = tokenizer(list(train_df["text"]), truncation=True, padding=True)
test_encodings = tokenizer(list(test_df["text"]), truncation=True, padding=True)

# 5. 入力・ラベルの整形（NumPy配列のまま分割 → 後でTensorに変換）
input_ids = np.array(train_encodings["input_ids"])
labels = np.array(train_df["target"].values)

X_train, X_val, y_train, y_val = train_test_split(input_ids, labels, test_size=0.1, random_state=42)

X_train = tf.convert_to_tensor(X_train)
X_val = tf.convert_to_tensor(X_val)
y_train = tf.convert_to_tensor(y_train)
y_val = tf.convert_to_tensor(y_val)

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(16)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(16)

# 6. モデルの構築とコンパイル
model = TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])

# 7. 学習（3エポック）
model.fit(train_dataset, validation_data=val_dataset, epochs=3)

# 8. テストデータで予測
X_test = tf.convert_to_tensor(test_encodings["input_ids"])
preds = model.predict(X_test).logits
pred_labels = np.argmax(preds, axis=1)

# 9. submission.csv の作成
submission = pd.DataFrame({
    "id": test_df["id"],
    "target": pred_labels
})
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("✅ submission.csv 作成完了！（3エポック + 前処理強化版）")
